# Section 02 lab — Reading data

Paths are relative to this section folder. Each exercise creates its own Spark session and reads or creates its own input.


## Exercise 1 — Compare default CSV types with inferSchema


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())

default_df = (spark.read.option('header', True)
              .csv('../data/transactions.csv'))
inferred_df = (spark.read.option('header', True)
               .option('inferSchema', True)
               .csv('../data/transactions.csv'))
print('Default CSV schema:')
default_df.printSchema()
print('Schema with inferSchema=True:')
inferred_df.printSchema()
print('Rows:', default_df.count())
print('Notice that malformed values can prevent a column from becoming the numeric type you expected.')


## Exercise 2 — Read CSV with an explicit schema


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql.types import StructType, StructField, StringType

transaction_schema = StructType([
    StructField('transaction_id', StringType(), True),
    StructField('transaction_date', StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('product_id', StringType(), True),
    StructField('quantity', StringType(), True),
    StructField('unit_price', StringType(), True),
    StructField('status', StringType(), True),
])
transactions = (spark.read.schema(transaction_schema)
                .option('header', True)
                .csv('../data/transactions.csv'))
transactions.printSchema()
transactions.show(5, truncate=False)
assert transactions.count() == 58


## Exercise 3 — Read JSON and TSV with explicit schemas


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

product_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('product_name', StringType(), True),
    StructField('category', StringType(), True),
    StructField('list_price', DoubleType(), True),
])
products_json = spark.read.schema(product_schema).json('../data/products.json')
products_tsv = (spark.read.schema(product_schema)
                .option('header', True).option('sep', '\t')
                .csv('../data/products.tsv'))
print('JSON products:', products_json.count())
print('TSV products:', products_tsv.count())
products_json.orderBy("product_id").show(5, truncate=False)
assert products_json.count() == products_tsv.count() == 12


## Exercise 4 — Create a DataFrame from Python rows and perform a Parquet round trip


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

rows = [('P101', 'Keyboard', 'Accessories', 40),
        ('P102', 'Monitor', 'Electronics', 12),
        ('P103', 'Notebook', 'Stationery', 100)]
inventory_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('product_name', StringType(), False),
    StructField('category', StringType(), False),
    StructField('stock', IntegerType(), False),
])
inventory = spark.createDataFrame(rows, inventory_schema)
output_path = '../output/section02_inventory_parquet'
inventory.write.mode('overwrite').parquet(output_path)
saved = spark.read.parquet(output_path)
saved.printSchema()
saved.orderBy("product_id").show()
assert saved.count() == 3
